In [1]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
from tqdm import tqdm

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['nature', 'no-latex'])

Functions

In [2]:
from sklearn.neighbors import NearestNeighbors

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)


def find_neighborhoods(x, n_neighbors: int = 50, metric: str = "minkowski", p: int = 2):
  k_nearest_neighbors = NearestNeighbors(
          n_neighbors=n_neighbors, metric=metric, p=p
      ).fit(x)
  _, ix_neighborhoods = k_nearest_neighbors.kneighbors(x)
  return ix_neighborhoods

def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx

def get_mask_and_valid_coordinate_i(reference_anat, mask_idx_list, jc, ir, mask_shape, coordinates):
  mask = np.zeros_like(reference_anat)
  mask_shape = mask.shape
  for mask_idx in mask_idx_list:
    start_idx = jc[mask_idx]
    end_idx = jc[mask_idx + 1]
    for i in ir[start_idx:end_idx]:
      i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
      mask[i_x, i_y, i_z] = 1
  valid_coordinate_i = []
  for i, c in enumerate(coordinates):
    if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
      valid_coordinate_i.append(i)
  valid_coordinate_i = np.array(valid_coordinate_i)
  return mask, valid_coordinate_i

In [3]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()
coordinates = coordinates.read().result()

f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

ValueError: NOT_FOUND: Error opening "zarr3" driver: Metadata at local file "/Users/s/vault/janelia/ts_files/subject_06_traces.zarr/zarr.json" does not exist [tensorstore_spec='{\"context\":{\"cache_pool\":{},\"data_copy_concurrency\":{},\"file_io_concurrency\":{},\"file_io_locking\":{},\"file_io_memmap\":false,\"file_io_sync\":true},\"driver\":\"zarr3\",\"kvstore\":{\"driver\":\"file\",\"path\":\"/Users/s/vault/janelia/ts_files/subject_06_traces.zarr/\"}}'] [source locations='tensorstore/driver/kvs_backed_chunk_driver.cc:1290\ntensorstore/driver/driver.cc:112']

In [ ]:
reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

# mask_idx_list = [106] # tectum neuropil
# mask_idx_list = [65] # pretectum
mask_idx_list = [108] # torus longitudinalis [almost all neurons that end here come from pretectum and tectum neuropil]
_, valid_coordinate_i = get_mask_and_valid_coordinate_i(reference_anat, mask_idx_list, jc, ir, mask_shape, coordinates)
valid_coordinate_i.shape

Try sbi with neural trace data as-is

In [ ]:
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE

_ = torch.manual_seed(0)

In [ ]:
valid_coordinate_i_h = np.mean(traces[:, valid_coordinate_i], 0) > 0.15

# valid_coordinate_i_h_l = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] < 300)
# valid_coordinate_i_h_r = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] > 300)

# valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_l]

traces_selected = traces[:, valid_coordinate_i[valid_coordinate_i_h]]
# traces_selected = traces[:, valid_coordinate_i_h]
traces_selected.shape

In [4]:
context = 256
n_d = context
horizon = 16
n_neurons = traces_selected.shape[-1]
traces_selected.shape[0]-context-horizon
trace_data_x = np.stack([traces_selected[i:i+context] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_theta = np.stack([traces_selected[i+context+horizon] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()

NameError: name 'traces_selected' is not defined

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::10])
trace_data_theta_train = torch.tensor(trace_data_theta[::10, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=4.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
# selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128] # tectum neuropil
# selected_ix = np.random.choice(np.arange(0, 90000), size=30, replace=False)
selected_ix = np.arange(0, valid_coordinate_i_h.shape[0])[valid_coordinate_i_h]
valid_coordinate_i_h = np.mean(traces[:, valid_coordinate_i], 0) > 0.15
selected_ix = valid_coordinate_i[valid_coordinate_i_h]
len(selected_ix)

In [ ]:
n_ix = 15
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
        theta_hat = posterior.sample((200,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)
fig.suptitle(f"Example traces and predictions, context {context}, horizon {horizon}, TL", fontsize=14, fontname="Arial")
plt.tight_layout()
plt.savefig(f'/Users/s/Documents/larvae/predictions_c{context}_h{horizon}_TL_active.png', bbox_inches='tight', dpi=500)


Now, we perform 3 experiments
- train on activity of the torus longitudinalis (TL) (where stimulus-locking is not as strong as in the pretectum (PT) and tectum neuropil (TN)), and connect selected sets of TL neurons to their k nearest neighbors
- train on activity of the TL and connect the highly active TL neurons to the highly active PT and TN neurons
- train on activity of the TL and connect selected sets of TL neurons to randomly selected neurons (control)

Load all neurons from TL

In [ ]:
mask_idx_list = [108]
mask, valid_coordinate_i = get_mask_and_valid_coordinate_i(mask, mask_idx_list, jc, ir, mask_shape, coordinates)
valid_coordinate_i.shape

Find the k-nn for each neuron based on their coordinates 

In [ ]:
def get_trace_data_train(traces: np.ndarray, coordinates: np.ndarray, valid_coordinate_i: np.ndarray, context: int, horizon: int, n_neighbors: int):
  nn_coordinate_i = find_neighborhoods(coordinates[valid_coordinate_i], n_neighbors=n_neighbors)
  traces_selected = traces[:, valid_coordinate_i][:, nn_coordinate_i].transpose(0, 2, 1)
  print(traces_selected.shape)
  n_neurons = traces_selected.shape[-1]
  trace_data_x = np.stack([traces_selected[i:i+context] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
  trace_data_x = trace_data_x.transpose(0, 3, 1, 2).reshape(trace_data_x.shape[0]*n_neurons, context*n_neighbors)
  trace_data_theta = np.stack([traces_selected[i+context+horizon, 0] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
  trace_data_theta = trace_data_theta.ravel()
  trace_data_x_train = torch.tensor(trace_data_x[::10])
  trace_data_theta_train = torch.tensor(trace_data_theta[::10, None])
  print(trace_data_x_train.shape, trace_data_theta_train.shape)
  return trace_data_x_train, trace_data_theta_train, traces_selected, nn_coordinate_i

In [ ]:
context = 140
horizon = 16
n_neighbors = 10

trace_data_x_train, trace_data_theta_train, traces_selected, nn_coordinate_i = get_trace_data_train(
    traces, coordinates, valid_coordinate_i, context, horizon, n_neighbors
)

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(context*n_neighbors), high=4.0 * torch.ones(context*n_neighbors))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
valid_coordinate_i_h = np.mean(traces_selected[:, 0, :], 0) > 0.13
traces_test = traces_selected[..., valid_coordinate_i_h]
traces_test.shape

In [ ]:
n_ix = 10
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i in tqdm(range(n_ix)):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces_test[..., i][j:j+context].ravel())
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces_test[:500+context, 0, i], 'k')
    ax.plot(np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

plt.tight_layout()

Now let's connect the active neurons in TL to the active neurons in PT and TN and try to predict the TL activity

In [ ]:
_, valid_coordinate_i_upstream = get_mask_and_valid_coordinate_i(reference_anat, [65, 106], jc, ir, mask_shape, coordinates)
_, valid_coordinate_i = get_mask_and_valid_coordinate_i(reference_anat, [108], jc, ir, mask_shape, coordinates)
valid_coordinate_i_upstream.shape, valid_coordinate_i.shape

In [ ]:
upstream_traces = traces[:, valid_coordinate_i_upstream][:, np.mean(traces[:, valid_coordinate_i_upstream], 0) > 0.20]
target_traces = traces[:, valid_coordinate_i][:, np.mean(traces[:, valid_coordinate_i], 0) > 0.13]
upstream_traces_mean = upstream_traces.mean(-1)
upstream_traces.shape, target_traces.shape

Plot upstream trace average and downstream activity

In [ ]:
n_ix = 15
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i in tqdm(range(n_ix)):
    ax = axs[i]
    ax.plot(target_traces[:1500, i], 'k')
    ax.plot(upstream_traces_mean[:1500], 'r')
    format_ax(ax)
fig.suptitle(f"Average upstream activity (TN, PT, active population) and TL traces", fontsize=14, fontname="Arial")
plt.tight_layout()

Create dataset

In [ ]:
def get_trace_data_train(upstream_traces_mean: np.ndarray, target_traces: np.ndarray, context: int, horizon: int):

  n_neurons = target_traces.shape[-1]
  trace_data_x = np.stack([np.concatenate([target_traces[i:i+context], np.broadcast_to(upstream_traces_mean[i:i+context, None], (context, n_neurons))], 0) for i in range(0, target_traces.shape[0]-context-horizon, 10)])
  print(trace_data_x.shape)
  trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context*2)

  trace_data_theta = np.stack([target_traces[i+context+horizon] for i in range(0, target_traces.shape[0]-context-horizon, 10)])
  trace_data_theta = trace_data_theta.ravel()

  trace_data_x_train = torch.tensor(trace_data_x[::1])
  trace_data_theta_train = torch.tensor(trace_data_theta[::1, None])
  print(trace_data_x_train.shape, trace_data_theta_train.shape)

  return trace_data_x_train, trace_data_theta_train

In [ ]:
context = 256
horizon = 1
trace_data_x_train, trace_data_theta_train = get_trace_data_train(upstream_traces_mean, target_traces, context, horizon)

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(context*2), high=4.0 * torch.ones(context*2))
inference = NPE(prior=prior)
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()
posterior = inference.build_posterior()

In [ ]:
traces_test = np.stack([target_traces, np.broadcast_to(upstream_traces_mean[:, None], (target_traces.shape[0], target_traces.shape[1]))], 1)

n_ix = 15
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i in tqdm(range(n_ix)):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces_test[j:j+context, :, i].ravel())
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces_test[:500+context, 0, i], 'k')
    ax.plot(np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context+horizon, theta_hat_i.shape[0]+context+horizon),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

plt.tight_layout()